In [10]:
import copy
import os
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
 
from models import *
 
checkpoint_path = "./result/vgg16_cifar10/model_best.pth.tar"   # path as you wrote it; fix spelling if the folder is "results"
EPOCHS, LR, BS = 10, 0.01, 128                        # fixed for every run so the comparison is fair
GAMMAS = [1e-5, 1e-3, 1e-1]                     # step 3 sweep, in addition to 0 (step 1) and 1 (step 2)
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])  # use HW3's values
train_set = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=transforms.Compose(
    [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor(), normalize]))
test_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor(), normalize]))
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BS, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
 
 
def build_pretrained():
    model = VGG16()                                       # same constructor as HW3
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['state_dict'])
    return model.to(device)
 
 
def first_conv(m):
    return next(mod for mod in m.modules() if isinstance(mod, nn.Conv2d))
 
 
def conv1_abs_sum(m):
    return first_conv(m).weight.abs().sum().item()
 
 
def test_acc(m):
    m.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            correct += (m(x.to(device)).argmax(1) == y.to(device)).sum().item()
    return 100.0 * correct / len(test_set)
 
 
def retrain(gamma):
    m = build_pretrained()
    conv1 = first_conv(m)
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(m.parameters(), lr=LR, momentum=0.9)   # no weight_decay, so the only shrinkage is loss2
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    for ep in range(EPOCHS):
        m.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss1 = criterion(m(x), y)
            loss2 = conv1.weight.abs().sum()
            loss = loss1 + gamma * loss2
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
        print(f"gamma={gamma:g} epoch {ep+1}/{EPOCHS} loss1 {loss1.item():.3f} "
              f"conv1 |W| sum {loss2.item():.2f}", flush=True)
    return m

In [11]:
results = []

m0 = build_pretrained()
results.append(("1. Initial (loss1)", "-", test_acc(m0), conv1_abs_sum(m0)))
print(results[-1])

('1. Initial (loss1)', '-', 94.03, 143.10568237304688)


In [12]:
m1 = retrain(1.0)
results.append(("2. loss1 + loss2", 1.0, test_acc(m1), conv1_abs_sum(m1)))
print(results[-1])

gamma=1 epoch 1/10 loss1 0.601 conv1 |W| sum 22.62
gamma=1 epoch 2/10 loss1 0.428 conv1 |W| sum 19.73
gamma=1 epoch 3/10 loss1 0.317 conv1 |W| sum 19.36
gamma=1 epoch 4/10 loss1 0.371 conv1 |W| sum 16.62
gamma=1 epoch 5/10 loss1 0.308 conv1 |W| sum 13.75
gamma=1 epoch 6/10 loss1 0.753 conv1 |W| sum 9.98
gamma=1 epoch 7/10 loss1 0.183 conv1 |W| sum 7.67
gamma=1 epoch 8/10 loss1 0.202 conv1 |W| sum 4.54
gamma=1 epoch 9/10 loss1 0.213 conv1 |W| sum 2.35
gamma=1 epoch 10/10 loss1 0.100 conv1 |W| sum 0.72
('2. loss1 + loss2', 1.0, 88.39, 0.7036581635475159)


In [13]:
models = {}
for g in GAMMAS:
    mg = retrain(g)
    models[g] = mg
    results.append((f"3. gamma sweep", g, test_acc(mg), conv1_abs_sum(mg)))
    print(results[-1])

gamma=1e-05 epoch 1/10 loss1 0.070 conv1 |W| sum 145.14
gamma=1e-05 epoch 2/10 loss1 0.052 conv1 |W| sum 147.48
gamma=1e-05 epoch 3/10 loss1 0.093 conv1 |W| sum 148.90
gamma=1e-05 epoch 4/10 loss1 0.084 conv1 |W| sum 149.42
gamma=1e-05 epoch 5/10 loss1 0.014 conv1 |W| sum 149.94
gamma=1e-05 epoch 6/10 loss1 0.004 conv1 |W| sum 149.77
gamma=1e-05 epoch 7/10 loss1 0.001 conv1 |W| sum 149.67
gamma=1e-05 epoch 8/10 loss1 0.001 conv1 |W| sum 149.49
gamma=1e-05 epoch 9/10 loss1 0.001 conv1 |W| sum 149.44
gamma=1e-05 epoch 10/10 loss1 0.002 conv1 |W| sum 149.43
('3. gamma sweep', 1e-05, 93.73, 149.42697143554688)
gamma=0.001 epoch 1/10 loss1 0.020 conv1 |W| sum 96.14
gamma=0.001 epoch 2/10 loss1 0.170 conv1 |W| sum 70.87
gamma=0.001 epoch 3/10 loss1 0.018 conv1 |W| sum 57.43
gamma=0.001 epoch 4/10 loss1 0.083 conv1 |W| sum 49.14
gamma=0.001 epoch 5/10 loss1 0.057 conv1 |W| sum 41.73
gamma=0.001 epoch 6/10 loss1 0.003 conv1 |W| sum 35.78
gamma=0.001 epoch 7/10 loss1 0.003 conv1 |W| sum 32.24
g

In [14]:
print(f"\n{'Case':<22}{'gamma':>8}{'Test acc (%)':>15}{'Conv1 |W| sum':>16}")
for name, g, acc, s in results:
    print(f"{name:<22}{str(g):>8}{acc:>15.2f}{s:>16.2f}")


Case                     gamma   Test acc (%)   Conv1 |W| sum
1. Initial (loss1)           -          94.03          143.11
2. loss1 + loss2           1.0          88.39            0.70
3. gamma sweep           1e-05          93.73          149.43
3. gamma sweep           0.001          93.31           25.11
3. gamma sweep             0.1          90.01            0.33
